<a href="https://colab.research.google.com/github/ecuirty-kr/1_DataAnalysis/blob/main/p11_type3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[구글 코랩(Colab)에서 실행하기](https://colab.research.google.com/github/lovedlim/bae-v3/blob/main/part4/ch11/p11_type3.ipynb)

## 작업형3

In [ ]:
### 정신(혼미)

In [13]:
### 문제 조건 : 제공된 데이터(elderly_health.csv)는 보건소에서 수집한 건강 검진 데이터로, 나이, 당뇨병 이력, 신체 활동 여부, 공복 형당 등에 따른
### 노령층 여부(predicted)에 관한 자료이다. 이 데이터를 Train 데이터(id: 1~2000)와 Test 데이터(id:2001~2227)로 나누어 로지스틱 회귀분석을 수행하고 답하시오.
### 제공 데이터 : 종속변수 - predicted, 노령층 여부(0= 비노령층, 1= 노령층) / 독립변수 - id, age, diabetic, activity, glus_fast, bmi, blood_pressure

## 문제 1. train 데이터에서 id 변수를 제외한 모든 변수를 활용하여 로지스틱 회귀분석을 수행. diabetic이 'no'인 사람 대비 'yes'인 사람의 오즈비(Odds Ratio)를 구하시오
## (반올림하여 소수 둘째 자리까지 작성)
## 문제 2. 위 모델에서 test 데이터 중 노령층일 확률이 가장 높은 사람의 glus_fast(공복 혈당) 값을 구하시오. (정수로 작성)
## 문제 3. 위 모델로 test 데이터를 예측한 후, 확률이 0.2 초과인 사람을 노령층으로 분류했을 때, 민감도(Sensivity)를 구하시오 (반올림하여 소수 둘째 자리까지 작성)

# 1. 라이브러리 및 데이터
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch11/elderly_health.csv")
# 데이터 확인
print(df.head(4))

## 문제 요구사항 : 이 데이터를 train, test 데이터로 나누어 로지스틱 회귀분석 수행
train = df[:2000].copy() # 데이터의 2000 까지를 추출하여 사본으로 train 나누기
test = df[2000:].copy() # 2000 부터 끝까지 test로 나누기

## 나눈 데이터로 로지스틱 회귀분석 수행 (id 제외 모든 변수 사용)
from statsmodels.formula.api import logit ## 라이브러리 잊지 말기
model = logit('predicted~age+diabetic+activity+glus_fast+bmi+blood_pressure', data=train).fit()
print(model.summary())

## 1-2. 오즈비 계산 (diabetic이 'no'인 사람 대비 'yes'인 사람의 오즈비) - 소수 둘째 반올림
## 오즈비 나오면? numpy
import numpy as np ## numpy 하면? np.exp()
odds_ratio = np.exp(model.params['diabetic[T.yes]'])
print(round(odds_ratio, 2)) # 결과 : 20.36
# 답안 : 20.36


## 문제 2. 위 모델에서 test 데이터 중 노령층일 확률이 가장 높은 사람의 glus_fast(공복 혈당)
## "가장 높은" = idxmax() 조합식 써야겠네 / 문제 뉘앙스상 예측 필요 = predict() + test데이터
# test 데이터로 예측 수행
pred = model.predict(test)  ## test 데이터로 노령층일 확률의 예측 수행

# 최대 확률을 가진 데이터 찾기
max_idx = pred.idxmax() ## pred : 예측 모델에서 / .idxmax() 최대 확률을 갖는 사람의 인덱스 값

# 최대 확률 데이터에서 glus_fast 값
print(test.loc[max_idx]) # 데이터가 출력됨. 여기서 glus_fast 값 찾아서 입력
# 답안 : 181  ******* 꼭 해당 값을 출력하려 하지 말고 해당 조건의 전체 데이터를 찾고 직접 확인할 수 있다는 점을 숙지할 것 **********


## 문제 3. 위 모델로 test 데이터를 예측, 확률이 0.2를 초과하는 사람을 노령층으로 분류했을 때, 민감도 구하기(소수 둘째 자리까지)
pred = model.predict(test) # test데이터 예측

# 조건 설정 : 확률이 0.2 초과 (>0.2)
pred = (pred>0.2).astype(int)  ## 저 조건식 (pred>0.2) 결과가 boolean 값으로 출력 -> .astype(int)로 정수형 변환

# 민감도 계산 (recall_score 사용) * 자매품 accuracy_score도 존재 (얘는 정확도) * 오류율은 (1 - 정확도)
from sklearn.metrics import recall_score
sensitivity = recall_score(test['predicted'], pred) ## recall_score(종속 변수 DF, 예측 pred) 인자값 숙지
print(round(sensitivity, 2)) # 결과 : 0.94
# 답안 : 0.94

   id  age diabetic activity  glus_fast   bmi  blood_pressure  predicted
0   1   70       no       no        154  26.8             136          1
1   2   58       no      yes        113  23.7             136          0
2   3   55       no       no        156  25.5             141          0
3   4   50       no      yes        183  29.8             100          0
Optimization terminated successfully.
         Current function value: 0.486192
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              predicted   No. Observations:                 2000
Model:                          Logit   Df Residuals:                     1992
Method:                           MLE   Df Model:                            7
Date:                Thu, 18 Jun 2026   Pseudo R-squ.:                  0.2883
Time:                        06:53:34   Log-Likelihood:                -972.38
converged:                       True   LL-Null:           

In [33]:
## 아 문제가 하나가 아니구나. 작업형 3번이.
## 문제 조건 : 제공된 데이터(promoton_data.csv)는 기업의 마케팅 홍보 활동에 따른 매출 데이터로, 방문 횟수, 웹페이지 방문 시간,
## 광고 클릭 수, 홍보 예산 등에 따른 매출액(sales)에 관한 자료이다. 이 데이터를 이용하여 통계 분석을 수행하고 답하시오.
## 제공 데이터 : 종속 변수 sales(매출액) / 독립 변수 visit_count, page_time, ad_clicks, promotion_budget

## 문제 1. 홍보 전후 매출 평균이 35,000원과 같은지 단일 t검정(t-test)을 수행하고, p-value를 구하시오 (소수 셋째 반올림)
## 바로 풀자. ttest_1smap 문제인데, 문제 조건이 "같은지" ttest_1samp(매출 평균, 35,000) * alternative (X)
import pandas as pd
from scipy.stats import ttest_1samp
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch11/promotion_data.csv")
# 데이터 확인
# print(df.head(5))

## 단일 표본 t-검정 수행 : 특정 조건이 '평균'이 조건과 같은지 -> 함수 자체에 평균을 담고 있으므로 평균 계산은 따로 안 함.
t_statistic, p_value = ttest_1samp(df['sales'], 35000)
print("p-value : ", round(p_value, 3)) # 결과 : p-value :  0.168
# 답안 : p-value :  0.168

## 문제 2. 매출액(sales)과 다른 모든 변수들 간의 피어슨 상관계수를 구하고, 그 중 상관관계가 가장 강한(절댓값 가장 큰) 상관계수 값을 구하시오
## 상관계수 = corr() / df에 corr()을 직접 적용하면 모든 변수간 상관 계수 확인 가능 -> sales 행렬 값 확인하여 가장 큰 값 구하기
print(df.corr()) # 가장 큰 값 :  0.575396
print(round( 0.575396, 3)) # 소수 셋째 반올림
# 답안 : 0.575

## 문제 3. 매출액(sales)을 종속변수로, 나머지 모든 변수를 독립변수로 하여 회귀분석 수행. 절편을 포함한 모든 변수 중 p-value가 가장 작은 변수의
## 회귀 계수를 구하기 (소수 셋째 반올림)

# 회귀 분석 수행 : ols
from statsmodels.formula.api import ols
model = ols('sales~visit_count+page_time+ad_clicks+promotion_budget', data=df).fit()
print(model.summary()) ## 회귀 분석 결과 확인

# 절편(Intercept) 포함 모든 변수 중 p-value 값이 가장 작은 변수
# print(model.pvalues) # ac_clicks 가 p-value 값이 가장 작음 (일단 확인은 함, 이렇게 확인하는게 맞나..?) => 아님.
## p-value 값이 가장 작은 데이터의 인덱스 추출
idx_min = model.pvalues.idxmin()
print("가장 작은 변수 : ", idx_min) # 결과 : 가장 작은 변수 :  page_time

# 회귀 계수 구하기 (회귀계수 : params[idx])
print(model.params[idx_min]) # 결과 : 130.73918703799623
print(round(model.params[idx_min], 3)) # 소수 셋째 반올림 : 130.739
# 답안 : 130.739

## 빅분기가 레전드다 진짜. 아 ..... .......

p-value :  0.168
                     sales  visit_count  page_time  ad_clicks  \
sales             1.000000     0.044154   0.575396  -0.131915   
visit_count       0.044154     1.000000   0.011349   0.078189   
page_time         0.575396     0.011349   1.000000  -0.093688   
ad_clicks        -0.131915     0.078189  -0.093688   1.000000   
promotion_budget -0.091270    -0.089217  -0.039554   0.036940   

                  promotion_budget  
sales                    -0.091270  
visit_count              -0.089217  
page_time                -0.039554  
ad_clicks                 0.036940  
promotion_budget          1.000000  
0.575
                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.343
Model:                            OLS   Adj. R-squared:                  0.330
Method:                 Least Squares   F-statistic:                     25.45
Date:                Thu, 18 Jun 2026   Prob (F-

### 문제1

In [ ]:
# 풀이1-1
import pandas as pd
import numpy as np
from statsmodels.formula.api import logit

df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch11/elderly_health.csv")

# 1) Train/Test 데이터 분할
train = df[:2000].copy()
test = df[2000:].copy()

# 2) 로지스틱 회귀분석 수행
model = logit("predicted ~ age + diabetic + activity + glus_fast + bmi + blood_pressure",
              data=train).fit()

# 3) 회귀 분석 결과 출력
print(model.summary())

# 4) diabetic='yes'의 오즈비 계산 및 출력
odds_ratio = np.exp(model.params["diabetic[T.yes]"])
print(round(odds_ratio, 2))

# 정답: 20.36

Optimization terminated successfully.
         Current function value: 0.486192
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              predicted   No. Observations:                 2000
Model:                          Logit   Df Residuals:                     1992
Method:                           MLE   Df Model:                            7
Date:                Sat, 03 Jan 2026   Pseudo R-squ.:                  0.2883
Time:                        06:11:30   Log-Likelihood:                -972.38
converged:                       True   LL-Null:                       -1366.3
Covariance Type:            nonrobust   LLR p-value:                7.486e-166
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -6.7577      0.754     -8.960      0.000      -8.236      -5.279
diabetic[T.pre

In [ ]:
# 풀이1-2
# 1) test 데이터 예측 확률 계산
pred = model.predict(test)

# 2) 최대 확률을 가진 데이터의 인덱스 찾기
max_idx = pred.idxmax()

# # 3) 해당 데이터의 glus_fast 값 출력
print(test.loc[max_idx])

# 정답: 181

id                2153
age                 78
diabetic           pre
activity            no
glus_fast          181
bmi               26.3
blood_pressure     115
predicted            1
Name: 2152, dtype: object


In [ ]:
# 풀이1-3
from sklearn.metrics import recall_score

# 1) test 데이터 예측
pred = model.predict(test)

# 2) 임계값 0.2 적용하여 분류
pred = (pred > 0.2).astype(int)

# 3) 민감도 계산 및 출력
sensitivity = recall_score(test['predicted'], pred)
print(round(sensitivity, 2))

# 정답: 0.94

0.94


### 문제2

In [ ]:
import pandas as pd
from scipy.stats import ttest_1samp

df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch11/promotion_data.csv")

# 1) 단일표본 t검정 수행
t_stat, p_value = ttest_1samp(df['sales'], 35000)

# 2) p-value 추출 및 출력
print(round(p_value, 3))

# 정답: 0.168

0.168


In [ ]:
# 1) 상관계수 행렬 계산
print(df.corr())

# 정답: 0.505

                     sales  visit_count  page_time  ad_clicks  \
sales             1.000000     0.044154   0.575396  -0.131915   
visit_count       0.044154     1.000000   0.011349   0.078189   
page_time         0.575396     0.011349   1.000000  -0.093688   
ad_clicks        -0.131915     0.078189  -0.093688   1.000000   
promotion_budget -0.091270    -0.089217  -0.039554   0.036940   

                  promotion_budget  
sales                    -0.091270  
visit_count              -0.089217  
page_time                -0.039554  
ad_clicks                 0.036940  
promotion_budget          1.000000  


In [ ]:
from statsmodels.formula.api import ols

# 1) ols 회귀분석 수행
model = ols("sales ~ visit_count + page_time + ad_clicks + promotion_budget", data=df).fit()

# 2) 회귀분석 결과 확인
print(model.summary())

# 3) p-value가 가장 작은 변수 찾기
min_var = model.pvalues.idxmin()
print("가장 유의한 변수:", min_var)

# 4) 해당 변수의 회귀계수 출력
coef = model.params[min_var]
print(round(coef, 3))

# 정답: 130.739

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.343
Model:                            OLS   Adj. R-squared:                  0.330
Method:                 Least Squares   F-statistic:                     25.45
Date:                Sat, 03 Jan 2026   Prob (F-statistic):           5.60e-17
Time:                        06:46:50   Log-Likelihood:                -2160.2
No. Observations:                 200   AIC:                             4330.
Df Residuals:                     195   BIC:                             4347.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept         2.234e+04   4136.622  